# Physics-guided histopathology experiments (UBC-OCEAN) - Colab runner

**Before you open this notebook (on your PC)**
1. In your project folder run (this packs only the filtered patches, max 16 per slide, into one zip):
   `python make_colab_zip.py --ubc-dir "D:\2026\Alliance university\Data Set\UBC Cancer subtype" --out "D:\2026\Alliance university\ubc_patches.zip" --max-per-slide 16`
2. In Google Drive create a folder **physics_paper** and upload into it: `ubc_patches.zip` and these 6 files:
   `ovarian_pipeline.py, novelty_utils.py, novel_model.py, histo_utils.py, histo_model.py, test_histo_utils.py`
3. Colab: *Runtime > Change runtime type > T4 GPU*, then run the cells below in order.

Results are written to Drive (`physics_paper/results_*`), so a disconnect loses at most one fold.

## 1. GPU check

In [ ]:
import subprocess, torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then re-run"

## 2. Mount Drive, copy code, unpack the patches to the fast local disk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, zipfile, time
DRIVE = "/content/drive/MyDrive/physics_paper"      # <- change if you used another folder name
CODE, DATA = "/content/code", "/content/data"
os.makedirs(CODE, exist_ok=True)
for f in ["ovarian_pipeline.py", "novelty_utils.py", "novel_model.py",
          "histo_utils.py", "histo_model.py", "test_histo_utils.py"]:
    shutil.copy(f"{DRIVE}/{f}", CODE)               # fails loudly if a file is missing
t = time.time()
with zipfile.ZipFile(f"{DRIVE}/ubc_patches.zip") as z:
    z.extractall(DATA)
UBC = f"{DATA}/ubc"
n = sum(len(fs) for _, _, fs in os.walk(UBC))
print(f"unpacked {n} files in {time.time() - t:.0f}s")
os.chdir(CODE)

## 3. Unit tests (about 1 minute). Expect `ALL TESTS PASSED`

In [ ]:
!python -W ignore test_histo_utils.py 2>&1 | tail -15

## 4. Quick smoke tests (a few minutes). First synthetic data, then a tiny real run

In [ ]:
!python histo_model.py --synthetic --epochs 2 --folds 3 --variants A_baseline E_full --no-pretrained --img-size 128 --out /content/smoke 2>&1 | tail -25

In [ ]:
!python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 4 --folds 3 --epochs 2 --variants A_baseline E_full --workers 2 --dl-workers 2 --out /content/check_histo 2>&1 | tail -40

## 5. Main run: baseline vs full model vs colour-jitter (the key comparison)
5-fold, slide-level CV. Expect roughly 1.5-2 h on a T4 (an estimate, not a measurement). Written to Drive.

In [ ]:
OUT_MAIN = f"{DRIVE}/results_main"
!python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 16 --folds 5 --epochs 10 --variants A_baseline E_full G_colorjitter --workers 2 --dl-workers 2 --out {OUT_MAIN}

## 6. Remaining ablation variants (run after cell 5; same seed and folds, so results are comparable)
Roughly another 2-3 h. Skip or run later if your session is short.
**Interrupted?** The command uses `--resume`: it continues after the last finished fold (it prints e.g. `resume: 3 of 5 folds already done`). Needs the updated `histo_utils.py` / `histo_model.py` in the Drive folder and cell 3 re-run. Just re-run this cell after a disconnect.


In [ ]:
OUT_REST = f"{DRIVE}/results_rest"
!python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 16 --folds 5 --epochs 10 --variants B_stainOD C_stainOD+consist D_attn_only F_consist_only --workers 2 --dl-workers 2 --resume --out {OUT_REST}

## 7. Merge all finished runs into one set of tables (safe to run with only cell 5 done)

In [ ]:
import json, sys, numpy as np
sys.path.insert(0, CODE)
import histo_utils as hu

res, cms = {}, {}
for d in [f"{DRIVE}/results_main", f"{DRIVE}/results_rest"]:
    if os.path.exists(f"{d}/results.json"):
        res.update(json.load(open(f"{d}/results.json")))
        if os.path.exists(f"{d}/confusion.json"):
            cms.update({k: np.array(v) for k, v in json.load(open(f"{d}/confusion.json")).items()})
order = ["A_baseline", "B_stainOD", "C_stainOD+consist", "D_attn_only", "F_consist_only", "G_colorjitter", "H_stainaug", "E_full"]
res = {k: res[k] for k in order if k in res}
cms = {k: cms[k] for k in order if k in cms}
df = hu.summarize_histo(res)
md_text = hu.to_markdown_histo(df, cms or None, hu.CLASSES if cms else None)
open(f"{DRIVE}/tables_all.md", "w", encoding="utf-8").write(md_text)
df.to_csv(f"{DRIVE}/summary_all_slide_level.csv", index=False)
hu.plot_robustness_histo(df, f"{DRIVE}/robustness_all.png")
print(md_text)

## 8. Stronger baseline + saved slide predictions (NEW code)
**First** upload the three updated files (`histo_utils.py`, `histo_model.py`, `test_histo_utils.py`) to the Drive folder, then re-run cell 3
(copy + unpack) so Colab uses them. Do this only after sections 5-6 have finished.

This run repeats A_baseline, G_colorjitter and E_full with the new code and adds **H_stainaug**: the same stain/illumination
perturbations the consistency loss uses, applied as plain augmentation (a like-for-like baseline without the physics layer).
It also saves out-of-fold slide predictions for bootstrap confidence intervals and adds a gentler blur test (`blur0.7`).
Roughly 1.5-2 h (an estimate).

In [ ]:
import os, shutil
OUT_STATS = f"{DRIVE}/results_stats"
os.makedirs(OUT_STATS, exist_ok=True)
if os.path.exists(f"{DRIVE}/results_main/radiomics_cache.npz"):            # reuse the cached radiomics (saves ~5 min)
    shutil.copy(f"{DRIVE}/results_main/radiomics_cache.npz", OUT_STATS)
!python test_histo_utils.py 2>&1 | tail -3
!python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 16 --folds 5 --epochs 10 --variants A_baseline G_colorjitter H_stainaug E_full --conditions clean clean_sub bright0.8 wb hscale1.3 escale0.7 blur blur0.7 jpeg30 noise --workers 2 --dl-workers 2 --resume --out {OUT_STATS}

## 9. Bootstrap confidence intervals and paired comparisons (needs section 8)

In [ ]:
import sys
sys.path.insert(0, CODE)
import histo_utils as hu
sp = hu.load_slide_predictions(f"{DRIVE}/results_stats/slide_predictions.csv")
variants = [v for v in ["A_baseline", "G_colorjitter", "H_stainaug", "E_full"] if v in set(sp.variant)]
text = "\n\n".join([hu.bootstrap_table_md(sp, variants, ref="A_baseline"),
                     hu.bootstrap_table_md(sp, variants, ref="G_colorjitter"),
                     hu.bootstrap_table_md(sp, variants, ref="H_stainaug", metric="macro_f1")])
open(f"{DRIVE}/bootstrap_tables.md", "w", encoding="utf-8").write(text)
print(text)

## What to bring back
`physics_paper/tables_all.md`, `robustness_all.png`, and from `results_main/`: `xai.json`, `xai_examples.png`.
Paste the tables here and we will interpret them and write the results / novelty text.
Also: `results_stats/tables.md`, `results_stats/robustness.png`, and `bootstrap_tables.md`.
